# LABORATORIO CALIFICADO N° 01
## Fundamentos de Gestión de Datos — Semana 2
**Curso:** Fundamentos de Gestión de Datos
**Docente:** Pilar Rocío Sayán Mejía
**Semestre:** 2026-I
**Modalidad:** Individual | Duración: 2 horas

---
## CASO: "El Caos de los Datos en RetailMax S.A."
**Protagonista:** Ana Torres, Analista de Datos Junior
**Empresa:** RetailMax S.A. — Cadena de tiendas por departamento con 12 sucursales en Perú

Ana acaba de incorporarse al área de Business Intelligence de RetailMax. Su primer encargo consiste en explorar los datos de ventas del primer trimestre 2026. Los datos llegan desde tres fuentes distintas con problemas de calidad.

**Las 3 preguntas que debe responder antes de la reunión del directorio:**
1. ¿Cuáles son los 5 productos más vendidos?
2. ¿Qué categoría genera mayor ingreso?
3. ¿Cuántos clientes compraron más de una vez?

---

## PASO 1: Generación de Datos Sintéticos de RetailMax

> Ejecute esta celda. Los datos representan las 3 fuentes del caso: ventas (CSV), clientes (SQLite) y productos (JSON). Se incluyen problemas intencionales de calidad.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import json
import matplotlib.pyplot as plt
import seaborn as sns
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

# ── Fuente 1: Datos de VENTAS (simula un CSV) ──────────────────
n_ventas = 300

categorias = ['Electronica', 'Ropa', 'Hogar', 'Alimentos', 'Deportes']
productos_por_cat = {
    'Electronica': ['Laptop', 'Celular', 'Tablet', 'Auriculares', 'Camara'],
    'Ropa':        ['Camisa', 'Pantalon', 'Vestido', 'Zapatillas', 'Chaqueta'],
    'Hogar':       ['Silla', 'Mesa', 'Lampara', 'Alfombra', 'Cojin'],
    'Alimentos':   ['Arroz', 'Aceite', 'Leche', 'Pan', 'Cafe'],
    'Deportes':    ['Pelota', 'Raqueta', 'Bicicleta', 'Mancuerna', 'Casco'],
}

cat_list = random.choices(categorias, weights=[20,25,20,20,15], k=n_ventas)
producto_list = [random.choice(productos_por_cat[c]) for c in cat_list]
precio_base = {'Electronica':500,'Ropa':80,'Hogar':150,'Alimentos':15,'Deportes':120}
precios = [precio_base[c] * round(random.uniform(0.8, 2.5), 2) for c in cat_list]
cantidades = [random.randint(1, 5) for _ in range(n_ventas)]
montos = [round(p * q, 2) for p, q in zip(precios, cantidades)]
ids_cliente = [f'C{random.randint(1,80):03d}' for _ in range(n_ventas)]

# Insertar problemas de calidad
montos_con_nulos = montos.copy()
for i in random.sample(range(n_ventas), 20):
    montos_con_nulos[i] = None  # nulos intencionales

ids_dup = ids_cliente.copy()
for i in range(5):  # duplicados intencionales
    ids_dup.append(ids_dup[i])
    cat_list.append(cat_list[i]); producto_list.append(producto_list[i])
    cantidades.append(cantidades[i]); montos_con_nulos.append(montos_con_nulos[i])

n_total = len(ids_dup)
fechas = [datetime(2026, 1, 1) + timedelta(days=random.randint(0, 89)) for _ in range(n_total)]

df_ventas = pd.DataFrame({
    'id_venta':    range(1, n_total + 1),
    'id_cliente':  ids_dup,
    'categoria':   cat_list,
    'producto':    producto_list,
    'cantidad':    cantidades,
    'monto_total': montos_con_nulos,
    'fecha':       [f.strftime('%Y-%m-%d') for f in fechas],
})

# ── Fuente 2: Datos de CLIENTES (simula SQLite) ───────────────
nombres = ['Luis','Maria','Carlos','Sofia','Jorge','Ana','Pedro','Rosa','Miguel','Carmen',
           'Diego','Lucia','Fernando','Isabella','Roberto','Valentina','Andres','Daniela']
apellidos = ['Torres','Garcia','Mendoza','Quispe','Vega','Cruz','Saenz','Llanos','Perez','Huaman']
ciudades = ['Lima','Arequipa','Cusco','Trujillo','Piura',None]  # None = nulo intencional

clientes = []
for i in range(1, 81):
    clientes.append({
        'id_cliente': f'C{i:03d}',
        'nombre':     f'{random.choice(nombres)} {random.choice(apellidos)}',
        'ciudad':     random.choices(ciudades, weights=[50,15,10,10,10,5])[0],
        'edad':       random.choice(list(range(18,65)) + [None, None]),  # nulos
        'segmento':   random.choice(['A','B','C']),
    })
df_clientes = pd.DataFrame(clientes)

# ── Fuente 3: Catálogo de PRODUCTOS (simula JSON) ─────────────
catalogo = []
for cat, prods in productos_por_cat.items():
    for prod in prods:
        catalogo.append({
            'nombre': prod,
            'categoria': cat,
            'precio_referencia': precio_base[cat],
            'activo': True,
        })
df_productos = pd.DataFrame(catalogo)

print("Datos generados exitosamente:")
print(f"  Ventas   : {len(df_ventas)} registros")
print(f"  Clientes : {len(df_clientes)} registros")
print(f"  Productos: {len(df_productos)} registros")


## PASO 2: Exploración Inicial del Dataset

> Ana necesita entender qué tiene antes de responder. Use las funciones de exploración de pandas.

In [ ]:
# === VENTAS ===
print("="*50)
print("TABLA DE VENTAS")
print("="*50)
print("\nPrimeras 5 filas:")
display(df_ventas.head())
print("\nInfo del DataFrame:")
df_ventas.info()
print("\nEstadísticas descriptivas:")
display(df_ventas.describe())


In [ ]:
# === CLIENTES ===
print("="*50)
print("TABLA DE CLIENTES")
print("="*50)
print("\nPrimeras 5 filas:")
display(df_clientes.head())
print("\nTipos de datos:")
print(df_clientes.dtypes)


In [ ]:
# === PRODUCTOS ===
print("="*50)
print("CATALOGO DE PRODUCTOS")
print("="*50)
display(df_productos)


## PASO 3: Detección de Problemas de Calidad

> Ana necesita saber con qué problemas se enfrenta antes de responder las preguntas del gerente.

In [ ]:
print("REPORTE DE CALIDAD DE DATOS")
print("="*50)

print("\n1. VALORES NULOS por columna:")
print("\n[Ventas]")
print(df_ventas.isnull().sum())
print("\n[Clientes]")
print(df_clientes.isnull().sum())

print("\n2. REGISTROS DUPLICADOS:")
dup_ventas = df_ventas.duplicated(subset=['id_cliente','categoria','producto','cantidad']).sum()
print(f"  Ventas con mismo cliente+producto+cantidad: {dup_ventas}")

print("\n3. TIPOS DE DATOS incorrectos o mixtos:")
print(df_ventas.dtypes)

print("\n4. RESUMEN DE CALIDAD:")
total = len(df_ventas)
nulos_monto = df_ventas['monto_total'].isnull().sum()
completitud = round((1 - nulos_monto/total)*100, 1)
print(f"  Total registros: {total}")
print(f"  Nulos en monto_total: {nulos_monto} ({100-completitud:.1f}%)")
print(f"  Completitud de monto: {completitud}%")
print(f"  Duplicados detectados: {dup_ventas}")


## PASO 4: Consultas SQL Básicas con SQLite

> Ana carga los datos en SQLite y usa SQL para responder las 3 preguntas del gerente.

In [ ]:
# Cargar datos en SQLite
conn = sqlite3.connect(':memory:')
df_ventas_limpio = df_ventas.dropna(subset=['monto_total']).copy()
df_ventas_limpio.to_sql('ventas', conn, if_exists='replace', index=False)
df_clientes.to_sql('clientes', conn, if_exists='replace', index=False)
df_productos.to_sql('productos', conn, if_exists='replace', index=False)
print("Tablas cargadas en SQLite exitosamente.")


In [ ]:
# PREGUNTA 1: Cuales son los 5 productos mas vendidos?
query1 = '''
SELECT producto,
       categoria,
       SUM(cantidad) AS total_unidades,
       COUNT(*) AS num_transacciones
FROM ventas
GROUP BY producto, categoria
ORDER BY total_unidades DESC
LIMIT 5
'''
resultado1 = pd.read_sql_query(query1, conn)
print("TOP 5 PRODUCTOS MAS VENDIDOS:")
display(resultado1)


In [ ]:
# PREGUNTA 2: Que categoria genera mayor ingreso?
query2 = '''
SELECT categoria,
       ROUND(SUM(monto_total), 2) AS ingreso_total,
       COUNT(*) AS num_ventas,
       ROUND(AVG(monto_total), 2) AS ticket_promedio
FROM ventas
GROUP BY categoria
ORDER BY ingreso_total DESC
'''
resultado2 = pd.read_sql_query(query2, conn)
print("INGRESOS POR CATEGORIA:")
display(resultado2)


In [ ]:
# PREGUNTA 3: Cuantos clientes compraron mas de una vez?
query3 = '''
SELECT COUNT(*) AS clientes_recurrentes
FROM (
    SELECT id_cliente
    FROM ventas
    GROUP BY id_cliente
    HAVING COUNT(*) > 1
)
'''
resultado3 = pd.read_sql_query(query3, conn)
print("CLIENTES QUE COMPRARON MAS DE UNA VEZ:")
display(resultado3)

query3b = '''
SELECT id_cliente,
       COUNT(*) AS num_compras,
       ROUND(SUM(monto_total), 2) AS gasto_total
FROM ventas
GROUP BY id_cliente
ORDER BY num_compras DESC
LIMIT 10
'''
resultado3b = pd.read_sql_query(query3b, conn)
print("\nTOP 10 CLIENTES POR FRECUENCIA:")
display(resultado3b)


## PASO 5: Visualización Básica

> Ana prepara dos gráficos para la presentación al directorio.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('RetailMax S.A. — Análisis de Ventas Q1 2026', fontsize=14, fontweight='bold')

# Gráfico 1: Histograma de montos de venta
axes[0].hist(df_ventas_limpio['monto_total'], bins=20, color='#2E75B6', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribución de Montos de Venta')
axes[0].set_xlabel('Monto Total (S/.)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(df_ventas_limpio['monto_total'].mean(), color='red', linestyle='--', label=f'Media: S/.{df_ventas_limpio["monto_total"].mean():.0f}')
axes[0].legend()

# Gráfico 2: Barras de ventas por categoría
ingresos_cat = df_ventas_limpio.groupby('categoria')['monto_total'].sum().sort_values(ascending=False)
colors = ['#1F3864','#2E75B6','#4472C4','#70AD47','#ED7D31']
axes[1].bar(ingresos_cat.index, ingresos_cat.values, color=colors)
axes[1].set_title('Ingresos Totales por Categoria')
axes[1].set_xlabel('Categoria')
axes[1].set_ylabel('Ingreso Total (S/.)')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(ingresos_cat.values):
    axes[1].text(i, v + 50, f'S/.{v:,.0f}', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('retailmax_analisis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graficos generados exitosamente.")


## PASO 6: Ficha Técnica del Dataset

In [ ]:
ficha = {
    "nombre_dataset": "RetailMax Q1 2026 - Datos de Ventas",
    "fuente_original": "Sistema interno RetailMax S.A. (datos sinteticos para laboratorio)",
    "fecha_extraccion": "2026-04-01",
    "tablas": {
        "ventas": {
            "registros": len(df_ventas),
            "columnas": list(df_ventas.columns),
            "tipos_principales": str(df_ventas.dtypes.to_dict()),
        },
        "clientes": {
            "registros": len(df_clientes),
            "columnas": list(df_clientes.columns),
        },
        "productos": {
            "registros": len(df_productos),
            "columnas": list(df_productos.columns),
        }
    },
    "dimensiones_calidad": {
        "completitud": f"{completitud}% (nulos en monto_total: {nulos_monto})",
        "unicidad": f"Duplicados detectados: {dup_ventas}",
        "exactitud": "Tipos de datos mixtos en columna 'fecha' (requiere conversion)",
        "consistencia": "Categorias consistentes entre ventas y catalogo",
        "oportunidad": "Datos del Q1 2026 - actualizacion trimestral"
    },
    "valor_de_negocio": [
        "Identificar productos estrella para optimizar inventario",
        "Segmentar clientes por frecuencia de compra para campanas de fidelizacion",
        "Analizar categorias con mayor retorno para decisiones de mix de productos"
    ]
}

print(json.dumps(ficha, ensure_ascii=False, indent=2))


## ACTIVIDAD 3: Análisis Reflexivo

**Instrucción:** Responda cada pregunta con mínimo 3 líneas. Argumente con evidencia de los datos explorados.

---

**A. ¿Qué tipo de datos (estructurado/semiestructurado/no estructurado) es cada fuente del caso? Justifique.**

*(Escriba su respuesta aquí)*

---

**B. ¿Cuál de las 3 preguntas del gerente fue más difícil de responder con SQL? ¿Por qué?**

*(Escriba su respuesta aquí)*

---

**C. Si usted fuera Ana Torres, ¿qué problema de calidad de datos atendería primero? ¿Por qué?**

*(Escriba su respuesta aquí)*

---

**D. ¿Qué valor de negocio aportan los datos explorados? ¿Qué decisión podría tomar RetailMax con esta información?**

*(Escriba su respuesta aquí)*


## CONCLUSIONES

Complete sus 3 conclusiones técnicas:

1. *(Conclusión 1 aquí)*

2. *(Conclusión 2 aquí)*

3. *(Conclusión 3 aquí)*

---
**Docente:** Pilar Rocío Sayán Mejía | **FGD 2026-I** | **LAB-C1**
